---
title: "Chapter 4: Crawling Results"
short_title: "4. Crawling Results"
---

This chapter reports the statistics required by the statement: the number of papers crawled successfully, the success rate, the average paper size before and after removing figures, the average number of references per paper, and the success rate of reference metadata crawling. All figures are recomputed from the final state stored on disk in each data folder, and every denominator is stated explicitly.

**Definitions used in this chapter:**

- **Successfully crawled paper:** every version is in a final state (`success`, `pdf_only` or `no_tex`, with its folder on disk and a matching file list), `metadata.json` is present, and the references are in the `success` or `empty` state. This is the metric `papers_pipeline_core_complete` of `corpus_statistics.py`.
- **Size before removing figures:** the total **uncompressed** size of all files in the source archive, summed over the versions of the paper. **Size after removing figures:** the total size of the kept `.tex`/`.bib`/`.bbl` files. Both sizes use the same basis and cover only papers whose versions are all in a final state.
- **After manual checks:** withdrawn versions (arXiv answers 404) and versions with an HTML source were confirmed on their arXiv abstract pages to have no TeX source ([Chapter 6](./06_errors_limitations.ipynb), Section 3). Table 4.1 includes two additional rows that treat these versions as final, in the same way as PDF-only versions; all other rows follow the definitions of `corpus_statistics.py`.
- **Average number of references:** the mean over the papers whose references are in the `success` or `empty` state, an `empty` paper counting as 0 references. Papers with API errors are not counted as 0.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

In [ ]:
papers = load("papers")
versions = load("versions")
m, g = members(), group()

found = papers[papers["discovery_status"] == "success"]
# assigned IDs: the official lists when exported with --assigned-ids, else the IDs of the manifests
assigned = bool(meta().get("assigned_ids"))
ids_total = int(metric("assigned_ids_expected")) if assigned else len(papers)
ids_missing = int(metric("assigned_ids_missing_from_manifest")) if assigned else None
core = metric("papers_pipeline_core_complete")
source_complete = metric("papers_source_complete")
strict = metric("papers_pipeline_complete_strict")

# sizes per paper, over papers whose every version reached a final state
final = versions["status"].isin(FINAL_SOURCE)
complete_ids = versions.assign(final=final).groupby("arxiv_id")["final"].all()
complete_ids = complete_ids[complete_ids].index
sizes = (versions[versions["arxiv_id"].isin(complete_ids)]
         .groupby(["member", "arxiv_id"])[["downloaded_bytes", "original_bytes", "image_bytes", "kept_bytes"]]
         .sum(min_count=1).fillna(0).reset_index())
with_source = sizes[sizes["original_bytes"] > 0]

# papers whose versions without a source were all checked by hand (withdrawn, HTML source)
checks = load("manual_checks")
checked = (versions["arxiv_id"] + versions["version"]).isin(set(checks["arxiv_id"] + checks["version"]))
by_paper = (pd.DataFrame({"arxiv_id": versions["arxiv_id"], "final": final, "final_or_checked": final | checked})
            .groupby("arxiv_id")[["final", "final_or_checked"]].all())
base = found[found["has_metadata"].astype(str).eq("True") & found["ref_status"].isin(REF_OK)]
core_recount = int(base["arxiv_id"].isin(by_paper.index[by_paper["final"]]).sum())
core_checked = int(base["arxiv_id"].isin(by_paper.index[by_paper["final_or_checked"]]).sum())

refs = found[found["ref_status"].isin(REF_OK)]
n_refs = refs["n_references"].sum()
n_arxiv = refs["n_with_arxiv_id"].sum()

rows = [
    ("Assigned IDs" if assigned else "Assigned IDs (from the manifests)", f"{ids_total:,}", "IDs"),
    ("Assigned IDs missing from the data", f"{ids_missing:,}" if assigned else "not checked", "IDs"),
    ("Papers existing on arXiv", f"{len(found):,}", "papers"),
    ("Papers crawled successfully", f"{core:,.0f}", "papers"),
    ("Success rate / existing papers", pct(core / len(found)), ""),
    ("Success rate / assigned IDs", pct(core / ids_total), ""),
    ("Papers crawled successfully after manual checks", f"{core_checked:,}", "papers"),
    ("Success rate after manual checks / existing papers", pct(core_checked / len(found)), ""),
    ("Papers with the source of every version", f"{source_complete:,.0f}", "papers"),
    ("Successful papers with the arXiv dates of all their references", f"{strict:,.0f}", "papers"),
    ("Version coverage (final state / versions to download)",
     pct(metric("versions_final_record_and_folder") / metric("versions_expected")), ""),
    ("Mean size per paper before removing figures (uncompressed)", human_bytes(with_source["original_bytes"].mean()), "per paper"),
    ("Mean size per paper after removing figures (.tex/.bib/.bbl)", human_bytes(with_source["kept_bytes"].mean()), "per paper"),
    ("Size reduction (1 − total kept / total original)",
     pct(1 - with_source["kept_bytes"].sum() / with_source["original_bytes"].sum()), ""),
    ("Mean compressed download size per paper", human_bytes(with_source["downloaded_bytes"].mean()), "per paper"),
    ("Mean references per paper", f"{n_refs / len(refs):,.2f}", f"references per paper, over {len(refs):,} papers"),
    ("Papers with references collected (success or empty) / existing papers", pct(len(refs) / len(found)), ""),
    ("References with complete metadata", pct(refs["n_complete"].sum() / n_refs), ""),
    ("References with an arXiv ID", pct(n_arxiv / n_refs), ""),
    ("References with an arXiv ID that have an arXiv submission date", pct(refs["n_with_ref_date"].sum() / n_arxiv), ""),
]
table(pd.DataFrame(rows, columns=["Metric", "Value", "Unit / denominator"]),
      "Table 4.1. Main statistics of the group")
if core_recount != core:
    print(f"Note: recounting from papers/versions gives {core_recount:,} successful papers, "
          f"which differs from {core:,.0f} in corpus_statistics.py.")

## 1. Per member

In [ ]:
def per_member(name, df):
    f = df[df["discovery_status"] == "success"]
    r = f[f["ref_status"].isin(REF_OK)]
    s = with_source[with_source["member"] == name]
    stats = load("statistics")
    core_m = pd.to_numeric(stats[(stats["member"] == name) & (stats["metric"] == "papers_pipeline_core_complete")]["value"]).sum()
    return {"Member": name, "IDs": len(df), "Existing papers": len(f), "Crawled successfully": int(core_m),
            "Success rate": pct(core_m / len(f)) if len(f) else "–",
            "Mean size before": human_bytes(s["original_bytes"].mean()),
            "Mean size after": human_bytes(s["kept_bytes"].mean()),
            "Mean references": f"{r['n_references'].mean():.2f}" if len(r) else "–",
            "Papers with references": pct(len(r) / len(f)) if len(f) else "–"}

table(pd.DataFrame([per_member(n, d) for n, d in papers.groupby("member")]),
      "Table 4.2. Main statistics per member")

## 2. Version status

Every version in the discovery list has a status; versions without TeX are recorded rather than dropped.

In [ ]:
st = versions.groupby(["member", "status"]).size().reset_index(name="versions")
fig = px.bar(st, x="member", y="versions", color="status", color_discrete_sequence=COLORS,
             title="Final status of the versions per member",
             labels={"member": "Member", "versions": "Versions", "status": "Status"})
show(fig)
total = versions["status"].value_counts().rename_axis("Status").reset_index(name="Versions")
total["Share"] = (total["Versions"] / total["Versions"].sum()).map(pct)
table(total, "Table 4.3. Final status of the versions, whole group")

## 3. Paper size before and after removing figures

The vertical axis uses a log scale because sizes span several orders of magnitude. Only papers with a TeX source (original size above 0) are included.

In [ ]:
long = with_source.melt(id_vars=["member", "arxiv_id"], value_vars=["original_bytes", "kept_bytes"],
                        var_name="kind", value_name="bytes")
long["kind"] = long["kind"].map({"original_bytes": "Before (all files, uncompressed)", "kept_bytes": "After (.tex/.bib/.bbl)"})
long = long[long["bytes"] > 0]
long["MB"] = long["bytes"] / 2**20
fig = px.box(long, x="kind", y="MB", color="kind", log_y=True, color_discrete_sequence=COLORS,
             title="Distribution of paper size before and after removing figures",
             labels={"MB": "Size per paper (MB, log scale)", "kind": ""})
fig.update_layout(showlegend=False)
show(fig)
ratio = 1 - with_source["kept_bytes"] / with_source["original_bytes"]
fig = px.histogram(x=ratio, nbins=50, color_discrete_sequence=COLORS,
                   title="Share of bytes removed per paper (1 − kept / original)")
fig.update_layout(showlegend=False, xaxis_title="Share removed", yaxis_title="Papers")
fig.update_xaxes(tickformat=".0%")
show(fig)
desc = pd.DataFrame({
    "Statistic": ["Mean", "Median", "90th percentile", "Maximum"],
    "Before removing figures": [human_bytes(x) for x in with_source["original_bytes"].agg(["mean", "median"]).tolist()
                                + [with_source["original_bytes"].quantile(0.9), with_source["original_bytes"].max()]],
    "After removing figures": [human_bytes(x) for x in with_source["kept_bytes"].agg(["mean", "median"]).tolist()
                               + [with_source["kept_bytes"].quantile(0.9), with_source["kept_bytes"].max()]]})
table(desc, f"Table 4.4. Size per paper, {len(with_source):,} papers with a TeX source")
print(f"Figure files alone account for {pct(with_source['image_bytes'].sum() / with_source['original_bytes'].sum())} "
      "of the original bytes;")
print("the remaining removed bytes are other files outside .tex/.bib/.bbl (classes, styles, data, ...).")
ok = versions[versions["status"] == "success"]
bbl = ok["kept_files"] - ok["tex_files"] - ok["bib_files"]
print(f"{(bbl > 0).sum():,} of the {len(ok):,} versions with a source ({pct((bbl > 0).mean())}) keep a .bbl file;")
print(f"{((bbl > 0) & (ok['bib_files'] == 0)).sum():,} of these have no accompanying .bib file.")

## 4. References

Reference crawl status of each paper (`missing` denotes a paper without a reference file, usually after an API error in the last run):

In [ ]:
rs = found["ref_status"].fillna("missing").value_counts().rename_axis("status").reset_index(name="papers")
fig = px.bar(rs, x="status", y="papers", text="papers", color_discrete_sequence=COLORS,
             title=f"Reference status of the {len(found):,} existing papers",
             labels={"status": "Status", "papers": "Papers"})
show(fig)
fig = px.histogram(refs, x="n_references", nbins=80, color_discrete_sequence=COLORS,
                   title=f"References per paper ({len(refs):,} papers with success/empty references)",
                   labels={"n_references": "References"})
fig.update_layout(yaxis_title="Papers")
show(fig)
q = refs.groupby("member")[["n_complete", "n_incomplete", "n_unresolved"]].sum().reset_index()
q = q.melt(id_vars="member", var_name="metadata_status", value_name="references")
q["metadata_status"] = q["metadata_status"].str.removeprefix("n_")
fig = px.bar(q, x="member", y="references", color="metadata_status", color_discrete_sequence=COLORS,
             title="Reference metadata quality per member",
             labels={"member": "Member", "references": "References", "metadata_status": "metadata_status"})
show(fig)